# DL Masterclass 02: Activation Functions & Gradient Dynamics

Welcome to Notebook 02. In the previous module, we learned that without a non-linear activation function, a 100-layer neural network mathematically collapses into a single $y = mx+b$ linear regression model.

But *which* non-linearity we choose dictates whether our gradients flow smoothly from Layer 100 back to Layer 1, or whether they die completely along the way.

---

## 🎓 Socratic Deep Check
Ponder this contradiction before we begin:

> *"If the ReLU function $\max(0, x)$ completely kills gradients for any negative input arriving at it (causing 'Dead Neurons' that never update again), why is it the most widely used activation function in the world instead of LeakyReLU, which specifically fixes this flaw?"*

---

## Table of Contents
1. **The Origin: Sigmoid & Tanh:** Why they cause the Vanishing Gradient problem.
2. **The King: ReLU:** Sparse representations and subgradient mathematics.
3. **Modern Era: GELU & Swish:** Why Transformers and LLMs abandoned ReLU for smoother approximations.


## 1. Sigmoid, Tanh, and the Vanishing Gradient

The Sigmoid function $\sigma(x) = \frac{1}{1 + e^{-x}}$ squashes all inputs into a range between `(0, 1)`. Tanh squashes them between `(-1, 1)`.

While excellent for outputting probabilities, they are terrible for deep hidden layers due to their derivatives.

### The Calculus of Death
The maximum possible value of the derivative of Sigmoid is `0.25` (occurring exactly at $x=0$). As you move away from 0, the derivative rapidly flattens to `0.0`.

Remember the **Chain Rule** from backpropagation. If you have a 10-layer network of Sigmoids, calculating the gradient for Layer 1 requires multiplying 10 derivatives together.

Even if you hit the absolute maximum derivative every time, the calculation is: $0.25 \times 0.25 \times 0.25 \dots = 0.25^{10} = 0.00000095$.

The error signal physically cannot reach the early layers. They stop learning. This is the **Vanishing Gradient Problem**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
sns.set_theme(style="whitegrid")

# -----------------------------------------------------
# IMPLEMENTATION: Visualizing Activation Derivatives
# -----------------------------------------------------

x = np.linspace(-5, 5, 200)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def d_sigmoid(x):
    s = sigmoid(x)
    return s * (1 - s)  # Maximum is 0.25

def tanh(x):
    return np.tanh(x)

def d_tanh(x):
    return 1 - np.tanh(x)**2  # Maximum is 1.0

def relu(x):
    return np.maximum(0, x)

def d_relu(x):
    return (x > 0).astype(float)  # Exactly 1.0 or 0.0

plt.figure(figsize=(14, 4))

# Plot Sigmoid
plt.subplot(1, 3, 1)
plt.plot(x, sigmoid(x), label="Sigmoid(x)")
plt.plot(x, d_sigmoid(x), '--', label="Derivative (Max 0.25!)", color='r')
plt.title("Sigmoid: The Vanishing Gradient")
plt.legend()

# Plot Tanh
plt.subplot(1, 3, 2)
plt.plot(x, tanh(x), label="Tanh(x)")
plt.plot(x, d_tanh(x), '--', label="Derivative (Max 1.0)", color='r')
plt.title("Tanh: Better, but still vanishes at tails")
plt.legend()

# Plot ReLU
plt.subplot(1, 3, 3)
plt.plot(x, relu(x), label="ReLU(x)")
plt.plot(x, d_relu(x), '--', label="Derivative (Always 1.0 or 0.0)", color='r')
plt.title("ReLU: Prevents Vanishing")
plt.legend()

plt.tight_layout()
plt.show()

## 2. ReLU & The Subgradient Problem

The Rectified Linear Unit (ReLU) $\max(0, x)$ solved the vanishing gradient problem. Since its derivative for any positive number is exactly $1.0$, multiplying a gradient by $1.0^{10}$ preserves the signal perfectly through 100 layers.

But what about exactly $x=0$? Mathematically, ReLU is broken there. A sharp 'V' shape does not have a defined tangent math derivative at the vertex. 

PyTorch solves this using **Subgradients**. It simply hardcodes a rule: `if x == 0: gradient = 0.0`. 

### 🎓 DEEP QUESTION ANSWERED
**Q:** *If ReLU causes Dead Neurons (0.0 gradient forever), why do we prefer it over LeakyReLU (which ensures a small gradient like 0.01 always flows)?*

**A:** **Sparsity.**

Biological brains are sparse; right now, 90% of your neurons are completely dormant. 

When a neural network feature representation consists of entirely dense, non-zero numbers, it is incredibly messy and entangled. By explicitly forcing 50% of the internal vector representations to an exact absolute $0.0$, ReLU enforces **Sparse Representations**. The remaining non-zero numbers become highly decoupled, clear signals. 

Furthermore, multiplying by exact $0.0$ in matrix multiplication is incredibly fast on GPU hardware (especially Sparse Matrix multipliers) compared to multiplying by `0.01` with LeakyReLU. We accept the rare risk of a permanently dead neuron in exchange for massive hardware efficiency and feature sparsity.

## 3. The Modern Era: Transformers & GELU

If ReLU is so great, why do OpenAI's GPT models, Meta's Llama, and almost all vision Transformers abandon ReLU in favor of **GELU** (Gaussian Error Linear Unit) or **Swish**?

`GELU(x) = x * P(X <= x)` where P is the standard Gaussian cumulative distribution.

It looks almost exactly like ReLU, but with two key differences:
1.  **It is infinitely smooth at 0.** No subgradient hacks required.
2.  **It dips slightly below zero.**

In highly complex, incredibly deep LLM architectures, the brutal "hard cutoff" of ReLU causes instability and blockiness in the optimization landscape. GELU provides a probabilistic, smooth off-ramp for neurons. The network can "softly" turn a neuron off without instantly terminating its gradient forever, leading to faster convergence in extreme-scale architectures.

In [ ]:
from scipy.stats import norm
import math

# -----------------------------------------------------
# IMPLEMENTATION: Visualizing GELU vs ReLU
# -----------------------------------------------------

x_zoom = np.linspace(-3, 3, 200)

# Exact GELU calculation
def gelu(x):
    return x * norm.cdf(x)

# Fast approximation of GELU (used in early BERT/GPT models to save compute)
def fast_gelu(x):
    return 0.5 * x * (1 + np.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x**3)))

plt.figure(figsize=(8, 5))
plt.plot(x_zoom, relu(x_zoom), label='ReLU (Hard Angle)', color='red', linestyle='--')
plt.plot(x_zoom, gelu(x_zoom), label='Exact GELU (Smooth & Convex-ish)', color='blue')
plt.plot(x_zoom, fast_gelu(x_zoom), label='Approx GELU (PyTorch default fallback)', color='green', linestyle=':')

plt.title("The LLM Standard: GELU vs ReLU")
plt.xlabel("Input (z)")
plt.ylabel("Activation Output")
plt.legend()
plt.grid(True)
plt.show()

# Notice the smooth parabolic dip below zero in GELU. This is the magic that
# stabilizers Transformers.